In [1]:
import sys

!{sys.executable} -m pip install \
langchain-community \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
sentence-transformers \
pypdf \
scikit-learn \
tiktoken

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

from langchain_groq import ChatGroq

import numpy as np

import tiktoken


class PDFChatbot:

    def __init__(self, pdf_path, groq_api_key):

        self.pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

        self.groq_api_key = "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"

        self.history = []

        self.token_limit = 3000

        self.load_pdf()

        self.create_embeddings()

        self.load_llm()

        self.tokenizer = tiktoken.get_encoding("cl100k_base")


    def load_pdf(self):

        loader = PyPDFLoader(self.pdf_path)

        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )

        self.chunks = splitter.split_documents(documents)

        self.texts = [
            doc.page_content for doc in self.chunks
        ]


    def create_embeddings(self):

        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        embeddings = self.embedding_model.embed_documents(
            self.texts
        )

        self.document_embeddings = np.array(embeddings)


    def load_llm(self):

        self.llm = ChatGroq(
            groq_api_key=self.groq_api_key,
            model_name="llama-3.3-70b-versatile"
        )


    def retrieve(self, query, k=3):

        query_embedding = self.embedding_model.embed_query(query)

        similarity_scores = cosine_similarity(
            [query_embedding],
            self.document_embeddings
        )[0]

        top_indices = similarity_scores.argsort()[::-1][:k]

        retrieved_chunks = [
            self.texts[i] for i in top_indices
        ]

        return retrieved_chunks


    def count_history_tokens(self):

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        tokens = self.tokenizer.encode(history_text)

        return len(tokens)


    def trim_history_if_needed(self):

        total_tokens = self.count_history_tokens()

        print(f"\nCurrent History Tokens: {total_tokens}")

        if total_tokens > self.token_limit:

            self.history = self.history[-8:]

            print(
                "\nHistory exceeded 3000 tokens."
            )

            print(
                "History trimmed to last 4 exchanges."
            )


    def ask(self, question):

        retrieved_docs = self.retrieve(question)

        context = "\n".join(retrieved_docs)

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )


        prompt = f"""
        You are a helpful PDF chatbot.

        Use the conversation history and document
        context to answer the question.

        Conversation History:
        {history_text}

        Context:
        {context}

        User Question:
        {question}
        """


        response = self.llm.invoke(prompt)

        answer = response.content


        self.history.append({
            "role": "user",
            "content": question
        })

        self.history.append({
            "role": "assistant",
            "content": answer
        })


        self.trim_history_if_needed()

        return answer


pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

groq_api_key = "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"


chatbot = PDFChatbot(
    pdf_path,
    groq_api_key
)


questions = [

    "What is the main topic of the PDF?",

    "Explain that topic in simple words.",

    "Which tools are mentioned?",

    "How are those tools used?",

    "Can you summarize everything discussed?"
]


for i, question in enumerate(questions, start=1):

    answer = chatbot.ask(question)

    print(f"\nTURN {i}")

    print("\nUSER:")
    print(question)

    print("\nBOT:")
    print(answer)

    print("\n" + "=" * 70)


print("\nFINAL CHAT HISTORY")


for message in chatbot.history:

    print(f"\n{message['role'].upper()}:")
    print(message['content'])

/var/folders/2q/c34xq8nn70lgm5zqqcds5x7m0000gn/T/ipykernel_24366/1628459157.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Current History Tokens: 46

TURN 1

USER:
What is the main topic of the PDF?

BOT:
The main topic of the PDF appears to be related to academic and student-related matters, specifically focusing on the student community, stress, and how students handle difficult situations.


Current History Tokens: 120

TURN 2

USER:
Explain that topic in simple words.

BOT:
The main topic of the PDF is about students and their lives. It looks at how students feel and how they deal with stress and tough situations. The survey found that overall, students are moderately to very happy with their daily lives. In simple words, it's about understanding the student community and how they handle challenges.


Current History Tokens: 189

TURN 3

USER:
Which tools are mentioned?

BOT:
Based on the conversation history and document context, there are no specific tools mentioned in the PDF. The context appears to be a survey focused on the student community, stress, and emotional well-being, with a section on h